The profiling script performs deep inspection of Gold layer tables to generate a centralized audit trail. It calculates volumetric and quality metrics to ensure data reliability for further analytics to be done on these tables.

**Key Features**
- Dynamic Table Discovery: Iterates through a JSON-defined list of datasets provided via Databricks widgets.
- Automated Audit Schema: Dynamically creates the audit schema if it does not exist within the Silver catalog.
- Volumetric Analysis: Captures total row counts, column counts, and unique record counts to verify deduplication success.
- Data Quality KPIs: Calculates total null counts and null_percent across all columns to flag data gaps.
- Incremental Auditing: Uses Delta Lake append mode with mergeSchema to maintain a historical log of data profiles over time.

In [0]:
import json
import warnings
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# Abstracted Imports
try:
    from state_mapping import (
        GOLD_CATALOG, 
        GOLD_SCHEMA, 
        AUDIT_SCHEMA, 
        PROFILE_FULL_PATH, 
        GOLD_METADATA_CONFIG, 
        DEFAULT_PROFILE_LIST
    )
except Exception:
    GOLD_CATALOG = 'data_gold'
    GOLD_SCHEMA = 'gold'
    AUDIT_SCHEMA = "audit"
    PROFILE_TABLE_NAME = "profile_summary"
    PROFILE_FULL_PATH = f"{GOLD_CATALOG}.{AUDIT_SCHEMA}.{PROFILE_TABLE_NAME}"
    GOLD_METADATA_CONFIG = {
    "dim_geography_gold": {
        "description": "Master Geography Dimension with hierarchical mapping of City, County, and State.",
        "columns": {"city": "Official city name."}
    },
    "dim_county": {
        "description": "County dimension for regional aggregation.",
        "columns": {"county": "Name of the county."}
    },
    "dim_state": {
        "description": "State dimension for high-level filtering.",
        "columns": {"state": "Full state name."}
    },
    "dim_region_type": {
        "description": "Lookup table for geographic grains (e.g., zip, city, county).",
        "columns": {"region_type": "The category of geographic grain."}
    },
    "dim_metric_dictionary": {
        "description": "Business Glossary mapping internal headers to human-readable definitions.",
        "columns": {"definition": "Official business logic definition."}
    },
    "fact_market_metrics_gold": {
        "description": "Enterprise Fact table for time-series market metrics.",
        "columns": {"date": "Measurement period."}
    }
    }
    DEFAULT_PROFILE_LIST = list(GOLD_METADATA_CONFIG.keys())

# --- 1. PARAMETER INITIALIZATION ---
# We derive the default widget value from state_mapping.DEFAULT_PROFILE_LIST
dbutils.widgets.text("datasets_json", json.dumps(DEFAULT_PROFILE_LIST), "Datasets List (JSON Array)")

def apply_enterprise_metadata(table_path, dataset_name):
    """
    Applies DDL comments. Logic is decoupled from table names.
    """
    metadata = GOLD_METADATA_CONFIG.get(dataset_name.lower())
    if not metadata:
        print(f"[INFO] No discovery metadata defined for {dataset_name}.")
        return

    try:
        spark.sql(f"COMMENT ON TABLE {table_path} IS '{metadata['description']}'")
        current_cols = spark.table(table_path).columns
        for col_name, comment in metadata['columns'].items():
            if col_name in current_cols:
                spark.sql(f"COMMENT ON COLUMN {table_path}.{col_name} IS '{comment}'")
        print(f"[SUCCESS] Metadata applied to {table_path}.")
    except Exception as e:
        if "STREAMING_TABLE_OPERATION_NOT_ALLOWED" in str(e):
            print(f"[SKIP] {dataset_name} is a DLT table; manual comments restricted.")
        else:
            print(f"[WARNING] Metadata error for {table_path}: {e}")

def profile_gold_tables():
    """
    Abstracted Profiling Engine.
    """
    # 1. Infrastructure Setup
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_CATALOG}.{AUDIT_SCHEMA}")
    
    # Explicit schema for the Audit table to prevent silent data type mismatches
    audit_schema = StructType([
        StructField("dataset_name", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("row_count", LongType(), True),
        StructField("column_count", LongType(), True),
        StructField("null_count", LongType(), True),
        StructField("null_percent", DoubleType(), True),
        StructField("unique_count", LongType(), True),
        StructField("columns", StringType(), True),
        StructField("profile_timestamp", TimestampType(), True)
    ])

    # 2. Input Resolution
    try:
        datasets_raw = dbutils.widgets.get("datasets_json")
        dataset_list = json.loads(datasets_raw)
    except Exception as e:
        print(f"[ERROR] Invalid widget input, falling back to defaults: {e}")
        dataset_list = DEFAULT_PROFILE_LIST

    all_profiles = []

    # 3. Processing Loop
    for ds in dataset_list:
        table_path = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.{ds.lower()}"
        
        try:
            if not spark.catalog.tableExists(table_path):
                print(f"[SKIP] Table {table_path} not found.")
                continue
            
            # Apply discovery metadata (descriptions)
            apply_enterprise_metadata(table_path, ds)
            
            print(f"[PROFILING] {ds}...")
            df = spark.table(table_path)
            row_count = df.count()
            col_count = len(df.columns)
            
            # Metric Calculation
            if row_count == 0:
                total_nulls, null_percent, unique_count = 0, 0.0, 0
            else:
                # Optimized single-pass null scanning
                null_counts_expr = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
                null_data = df.select(null_counts_expr).collect()[0].asDict()
                total_nulls = sum(null_data.values())
                null_percent = (total_nulls / (row_count * col_count)) * 100
                unique_count = df.dropDuplicates().count() 
            
            all_profiles.append({
                "dataset_name": ds,
                "layer": "GOLD",
                "row_count": row_count,
                "column_count": col_count,
                "null_count": total_nulls,
                "null_percent": round(float(null_percent), 2),
                "unique_count": unique_count,
                "columns": ", ".join(df.columns),
                "profile_timestamp": datetime.now()
            })
            
        except Exception as e:
            print(f"[ERROR] Could not profile {ds}: {str(e)}")

    # 4. Atomic Write to Audit Table
    if all_profiles:
        profile_df = spark.createDataFrame(all_profiles, schema=audit_schema)
        (profile_df.write.format("delta")
                  .mode("append")
                  .option("mergeSchema", "true")
                  .saveAsTable(PROFILE_FULL_PATH))
        print(f"[SUCCESS] Profiling stored in {PROFILE_FULL_PATH}.")
    else:
        print("[WARNING] No profiling data generated.")

if __name__ == "__main__":
    profile_gold_tables()

In [0]:
# Arguments to be passed here from job and as task params in the job
#["dim_county","dim_state", "dim_metric_dictionary", "dim_region_type","dim_geography_gold","fact_market_metrics_gold"]

# Since MVs don't have data stored in explicitly, running profiling on them won't give correct numbers but it will provide correct ones for objects which have data stored in them like Streaming tables and normal tables